# modular-diffusion: the first capture, one noise band's engine made from Stable Diffusion 1.5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/modular-diffusion/blob/v0.9.1/colab/first_capture.ipynb)

The capture library makes one smaller U-Net (a band engine) per stretch of the noise range from Stable Diffusion 1.5 (`geolip.modular_diffusion.capture`). This notebook makes the engine of the CLEAN band (steps 0 to 409), with no prompt input (it reads a constant context of its own), three ways on the shape `tiny` (one layer a stage, no middle block, three stages): the model's own weights inherited and trained on the model's answers; random weights trained on the model's answers; random weights trained on the noise itself. The inherited way also runs on the larger shape `2222/nomid/3`. Two seeds each, 3,000 steps at batch 16, pure Adam, an averaged copy used for every read. Every 250 steps the live engine and its averaged copy are read on 64 held-out rows (never trained on). Each finished engine is then served in the sampler beside the model (the model keeps the middle and noisy bands; 50 DDIM steps, guidance 7.5), exchanged for its twin of the other seed, and served in the middle band instead (the control that must be worse).

1. install at a release tag and print the environment;
2. the model and both COCO val2017 parquet files at pinned revisions; the probe latents (the first 256 pictures, with their content hash checked) and the training latents (the other 4,744 pictures); a smoke run of every job kind on a tiny random model, with its exactness checks; the sampler checked against the diffusers library's own DDIM scheduler on the real model; the smoke shipped (so the token and the upload are checked before the long run); a timing of two shapes; the job list with its projected minutes;
3. the job list, one job at a time; each finished job folder is shipped at once with its engine files; a progress line a minute; the summary; the machine is given back only when every finished job and the report were shipped.

WEIGHTS: the trained engines are derivatives of Stable Diffusion 1.5. They are stored beside the tables in the results repository (each training's folder holds `engine.safetensors`, `engine_avg.safetensors` and `engine.json`), with a copy of that model's license: before any trained weights are uploaded, C2 places a copy of the CreativeML Open RAIL-M license and a notice saying what these files are at the root of the results folder. The checkpoint used for resuming is not shipped.

RESUME: run the cells again. Finished jobs keep their markers under `OUT`. On a new machine, finished jobs are fetched back from the Hub with their engine files, so nothing trains twice.

In [ ]:
# C1 - install at a tag (extras on the name); torch is not a dependency, so the runtime's torch stays
%pip install -q "modular-diffusion[capture,hub,data,guard] @ git+https://github.com/AbstractEyes/modular-diffusion@v0.9.1"
import os, sys, time, json, gc, hashlib, shutil, statistics, threading
import torch, geolip.modular_diffusion as MD
assert tuple(int(x) for x in MD.__version__.split(".")) >= (0, 9, 1), "RESTART REQUIRED: Runtime > Restart session, then run C1 again"
import diffusers, transformers, huggingface_hub
print("torch", torch.__version__, "| diffusers", diffusers.__version__, "| transformers", transformers.__version__,
      "| huggingface_hub", huggingface_hub.__version__, "| package", MD.__version__)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("card:", props.name, f"| memory {props.total_memory / 2**30:.1f} GiB | cpus: {os.cpu_count()}")
else:
    print("card: NONE (choose a GPU runtime)")
OUT = "/content/first_capture"       # the output root; keep it the same to resume
os.makedirs(OUT, exist_ok=True)

The model comes from the Hub at a pinned revision, and the pictures from both validation parquet files of COCO 2017 with five captions each (the first is used). The PROBE is the first 256 pictures of the first file, encoded exactly as in the band-needs notebook (its content hash must match); the reads use its first 64 rows. The TRAINING latents are every other picture (rows 256 on of the first file, then the whole second file): 4,744 pictures, decoded and encoded 512 at a time so memory stays flat. Both are written once and skipped when present. Then: every job kind of the list runs for a few steps on a tiny random model (an assembly of the model's own parts must change the final latents by exactly 0, and an engine exchanged for itself must change nothing); the sampler is compared with the diffusers library's DDIM scheduler on the real model; the smoke folder is shipped without weights; two shapes are timed; the job list is sized from those times.

In [ ]:
# C2 - the model, the data, the checks, the size of the job list
from huggingface_hub import snapshot_download, hf_hub_download
import pyarrow.parquet as pq
from geolip.modular_diffusion import hub, run
from geolip.modular_diffusion.capture.assemble import initial_noise, sample
from geolip.modular_diffusion.capture.data import (LatentBatches, center_crop, encode_shards, parquet_pictures, read_index,
                                                   rows_in_order, synthetic_shards)
from geolip.modular_diffusion.capture.teacher import load_teacher
from geolip.modular_diffusion.capture.train import train_band_engine
from geolip.modular_diffusion.experiments import capture
TEACHER, PROBE, TRAIN, RUNS = f"{OUT}/sd15", f"{OUT}/probe", f"{OUT}/train", f"{OUT}/runs"
REPO = "AbstractPhil/geolip-multirotary-diffusion-collective"   # the results repository ("" = nothing shipped)
PREFIX = "modular_diffusion/capture/first_capture"
NOT_SHIPPED = ("checkpoint.pt", "SHIPPED", "FROM_HUB")           # the resume checkpoint and two markers; the engine files ship
os.makedirs(RUNS, exist_ok=True)

# the data
snapshot_download(repo_id="stable-diffusion-v1-5/stable-diffusion-v1-5", revision="451f4fe16113bff5a5d2269ed5ad43b0592e9a14",
                  allow_patterns=["model_index.json", "scheduler/scheduler_config.json", "tokenizer/*", "text_encoder/config.json",
                                  "text_encoder/model.safetensors", "unet/config.json", "unet/diffusion_pytorch_model.safetensors",
                                  "vae/config.json", "vae/diffusion_pytorch_model.safetensors"], local_dir=TEACHER)
PARQUETS = [hf_hub_download(repo_id="lmms-lab-encoder/COCO-Caption2017", repo_type="dataset", revision="3bdd5827e243cc3084ac69a1111e69c3ab9193ff",
                            filename=f"data/val-0000{i}-of-00002.parquet", local_dir=f"{OUT}/coco") for i in (0, 1)]
N_PROBE = 256

def content_hash(pictures, captions, size=512):
    h = hashlib.sha256()
    for p, c in zip(pictures, captions):
        a = ((center_crop(p, size) + 1.0) * 127.5).round().clamp(0, 255).to(torch.uint8)    # the array the encoder sees
        h.update(a.numpy().tobytes())
        h.update(c.encode("utf-8") + b"\0")
    return h.hexdigest()

def first_captions(path):
    out = []        # the first caption of every row, read without the pictures (the rule of parquet_pictures)
    for b in pq.ParquetFile(path).iter_batches(batch_size=512, columns=["answer"]):
        for a in b.column(0).to_pylist():
            out.append("" if a is None else str((a[0] if a else "") if isinstance(a, (list, tuple)) else a))
    return out

EXPECTED_CONTENT_HASH = "93d333a65f7bbdac6ced9143ff55852273c4e2a64deb7f8f641aeb1878f35c44"           # the 256 probe pictures of the band-needs run
SOURCES = [(PARQUETS[0], N_PROBE), (PARQUETS[1], 0)]   # the training pictures: (file, first row)

def train_pictures():
    # 512 pictures at a time (decoded, handed to the encoder one by one, then dropped), so memory stays flat
    for path, skip in SOURCES:
        total = pq.ParquetFile(path).metadata.num_rows
        for first in range(skip, total, 512):
            pics, _, _ = parquet_pictures(path, min(512, total - first), skip=first)
            yield from pics
            del pics

probe_ready = os.path.exists(f"{PROBE}/index.json") and sum(read_index(PROBE)["counts"]) == N_PROBE
train_captions = [c for path, skip in SOURCES for c in first_captions(path)[skip:]]
TRAIN_ROWS = len(train_captions)
print("training pictures:", TRAIN_ROWS, "| expected 4,744")
assert TRAIN_ROWS == 4744, "the training rows are not the expected ones"
train_ready = os.path.exists(f"{TRAIN}/index.json") and sum(read_index(TRAIN)["counts"]) == TRAIN_ROWS
if not (probe_ready and train_ready):
    with_vae = load_teacher(TEACHER, device="cuda", dtype=torch.float32, vae=True)
    if not probe_ready:
        pictures, captions, meta = parquet_pictures(PARQUETS[0], N_PROBE)
        HASH = content_hash(pictures, captions)
        print("probe content hash", HASH, "| expected", EXPECTED_CONTENT_HASH)
        assert HASH == EXPECTED_CONTENT_HASH, "the probe pictures are not the expected data"
        encode_shards(pictures, captions, PROBE, with_vae.vae, with_vae.tokenize, shard_size=64, batch=16, size=512, device="cuda")
        json.dump({"content_hash": HASH, "pictures": [{"row": i, "id": m.get("id"), "license": m.get("license")} for i, m in enumerate(meta)]},
                  open(f"{PROBE}/pictures.json", "w", encoding="utf-8"), indent=1)
        del pictures
    if not train_ready:
        # encode_shards is given a generator of pictures; a stopped encoding continues at its first unfinished shard
        encode_shards(train_pictures(), train_captions, TRAIN, with_vae.vae, with_vae.tokenize, shard_size=512, batch=16, size=512,
                      device="cuda")
    del with_vae
    gc.collect()
    torch.cuda.empty_cache()
print("probe shards:", read_index(PROBE)["counts"], "| training shards:", read_index(TRAIN)["counts"])

# the job list: (tag, arguments, the jobs it waits for, kind); the folders it reads are under `runs`
COMMON = (f"teacher={TEACHER} device=cuda threads=4 probe_data={PROBE} probe_first=0 probe_rows=64 probe_seed=1000 "
          "probe_per_band=4 chunk=16")
TRAINING = f"variant=band_engine band=clean text=0 steps=3000 batch=16 lr=5e-5 autocast=bf16 read_every=250 keep_checkpoint=0 dtype=fp32 data={TRAIN}"
SAMPLING = "sample_steps=50 guidance=7.5 group=16 dtype=fp16"     # the teacher in half precision; the engines stay float32

def job_list(runs):
    at = lambda t: f"{runs}/{t}"
    jobs = [("born_full_notext", "variant=born shape=full text=0 dtype=fp32", [], "born"),
            ("born_tiny_text", "variant=born shape=tiny text=1 dtype=fp32", [], "born"),
            ("asm_teacher", f"variant=assembly parts=clean=teacher,middle=teacher,noisy=teacher {SAMPLING}", [], "sample")]
    for name, shape, init, extra, more in (("tiny_inherit", "tiny", "inherit", "", True),
                                           ("2222nomid3_inherit", "2222/nomid/3", "inherit", "", True),
                                           ("tiny_random", "tiny", "random", "", False),
                                           ("tiny_random_noise", "tiny", "random", " lam_out=0 lam_data=1", False)):
        t = [f"clean_{name}_s{s}" for s in (0, 1)]
        jobs += [(t[s], f"{TRAINING} shape={shape} init={init} seed={s}{extra}", [], "train") for s in (0, 1)]
        jobs += [(f"asm_{t[s]}", f"variant=assembly parts=clean={at(t[s])},middle=teacher,noisy=teacher {SAMPLING}", [t[s]], "sample")
                 for s in (0, 1)]
        if more:
            jobs += [(f"off_{t[s]}", f"variant=assembly parts=clean=teacher,middle={at(t[s])},noisy=teacher {SAMPLING}", [t[s]], "sample")
                     for s in (0, 1)]
            jobs += [(f"xchg_{name}", f"variant=exchange parts=clean={at(t[0])},middle=teacher,noisy=teacher band=clean "
                      f"other={at(t[1])} {SAMPLING}", t, "exchange")]
    return jobs

# the smoke: every job kind of the list on the tiny random model, a few steps each, with the exactness keys
DEVICE = "cuda"
SMOKE = f"{OUT}/smoke"
shutil.rmtree(SMOKE, ignore_errors=True)
synthetic_shards(f"{SMOKE}/rows", rows=8, shard_size=3, size=16, seed=4)       # the tiny model's 16 x 16 latents
SMALL = (f" tiny=1 device={DEVICE} probe_data={SMOKE}/rows probe_rows=4 probe_per_band=2 chunk=2 steps=4 batch=2 read_every=2"
         " sample_steps=4 group=2 save_every=0 log_every=0" + ("" if DEVICE == "cuda" else " dtype=fp32 autocast=none"))
smoke_jobs = job_list(f"{SMOKE}/runs")
smoke_jobs.append(("xchg_self", f"variant=exchange parts=clean={SMOKE}/runs/clean_tiny_inherit_s0,middle=teacher,noisy=teacher "
                   f"band=clean other={SMOKE}/runs/clean_tiny_inherit_s0 {SAMPLING}", ["clean_tiny_inherit_s0"], "exchange"))
S = {}
for t, args, needs, kind in smoke_jobs:
    S[t] = run.main("capture", f"{args} {COMMON}{SMALL}".split() + [f"out={SMOKE}/runs/{t}"])
assert S["asm_teacher"]["final_latent_change"] == 0.0, "an assembly of the model's own parts changed the final latents"
x = S["xchg_self"]["exchange"]
assert x["final_latent_change"] == 0.0 and all(v["change"] == 0.0 and v["largest_change_where_silent"] == 0.0
                                               for v in x["bands"].values()), "an engine exchanged for itself changed something"
assert all(v["largest_change_where_silent"] == 0.0 for t in ("xchg_tiny_inherit", "xchg_2222nomid3_inherit")
           for v in S[t]["exchange"]["bands"].values() if v["silent_steps"]), "an exchange moved a band where the clean band is silent"
assert all(r["mse"] == 0.0 for t in ("born_full_notext", "born_tiny_text") for r in S[t]["rows"] if r["part"] == "teacher copy")
assert all(len(S[t]["curve"]) == 3 for t, _, _, k in smoke_jobs if k == "train"), "a training has no read curve"
print(f"smoke: {len(smoke_jobs)} jobs of every kind ran on the tiny model; the exactness keys hold")

# the sampler against the diffusers library's own DDIM scheduler, on the real model (4 probe rows, float32), JUDGED STEP BY
# STEP: at each of the 50 steps both make one step from the same latent and must agree to rounding. The two full
# trajectories are compared as well, beside a control: the sampler against itself from a start noise changed by one part
# in ten million. The model with guidance amplifies rounding from step to step, so the full trajectories of two correct
# samplers drift apart; they must stay closer than the control's.
import math
from diffusers import DDIMScheduler
from geolip.modular_diffusion.capture.assemble import ddim_timesteps, guided
from geolip.modular_diffusion.capture.engine import as_part
from geolip.modular_diffusion.capture.schedule import sd15_alphas_cumprod
teacher = load_teacher(TEACHER, device=DEVICE, dtype=torch.float32)
x0, ids = rows_in_order(PROBE, 0, 4)
cond, empty = teacher.text_states(ids), teacher.empty_states(1)
shape = (int(teacher.unet.config.in_channels), int(teacher.unet.config.sample_size), int(teacher.unet.config.sample_size))
part, ac = as_part(teacher), sd15_alphas_cumprod().double()
sch = DDIMScheduler(num_train_timesteps=1000, beta_start=0.00085, beta_end=0.012, beta_schedule="scaled_linear", steps_offset=1,
                    set_alpha_to_one=False, clip_sample=False, prediction_type="epsilon")
sch.set_timesteps(50)
ts = ddim_timesteps(50, 1)
assert [int(t) for t in sch.timesteps] == ts, "the two samplers visit different steps"
both_states = torch.cat([empty.expand(4, -1, -1), cond])

def our_step(z, t):
    # one step of the library's sampler, written out with its own arithmetic (assemble.sample)
    tt = torch.full((4,), t, dtype=torch.long, device=z.device)
    eps = guided(part, z, tt, cond, empty, 7.5)
    a, prev = float(ac[t]), t - 1000 // 50
    a_prev = float(ac[prev]) if prev >= 0 else float(ac[0])
    x0h = (z - math.sqrt(1.0 - a) * eps) / math.sqrt(a)
    return math.sqrt(a_prev) * x0h + math.sqrt(1.0 - a_prev) * eps

def their_step(z, t):
    both = teacher.unet(torch.cat([z, z]), t, encoder_hidden_states=both_states).sample
    u, c = both[:4], both[4:]
    return sch.step(u + 7.5 * (c - u), t, z, eta=0.0).prev_sample

apart = lambda a, b: float((a.double() - b.double()).norm() / b.double().norm())
z0 = initial_noise(1000, 4, shape).to(DEVICE)
with torch.no_grad():
    whole = sample(teacher, cond, empty, 1000, 50, 7.5, latent_shape=shape)
    za, zb, one_step = z0.clone(), z0.clone(), []
    for t in ts:
        a1 = our_step(za, t)
        one_step.append(apart(a1, their_step(za, t)))       # one step each, from the same latent
        zb = their_step(zb, t)                               # the scheduler's own trajectory
        za = a1
    g = torch.Generator().manual_seed(7)
    zc = z0 + 1e-7 * torch.randn(z0.shape, generator=g).to(DEVICE) * z0
    for t in ts:
        zc = our_step(zc, t)
PARITY = {"relative_difference": max(one_step), "one_step_first": one_step[0], "one_step_last": one_step[-1],
          "trajectory_difference": apart(za, zb), "control_start_changed_by_1e-7": apart(zc, za),
          "written_out_step_against_the_library_sampler": apart(za, whole), "rows": 4, "steps": 50, "guidance": 7.5,
          "dtype": "fp32", "card": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}
json.dump(PARITY, open(f"{RUNS}/parity.json", "w", encoding="utf-8"), indent=1)
print(f"the sampler against diffusers' DDIM scheduler on the real model: one step from the same latent, largest relative "
      f"difference over the 50 steps {PARITY['relative_difference']:.3e}; the two full trajectories end "
      f"{PARITY['trajectory_difference']:.3e} apart; the control (a start changed by 1e-7) ends "
      f"{PARITY['control_start_changed_by_1e-7']:.3e} apart; the written-out step against the library's sampler "
      f"{PARITY['written_out_step_against_the_library_sampler']:.3e}")
assert PARITY["relative_difference"] < 1e-5, "one step of the sampler does not agree with the DDIM scheduler"
assert PARITY["trajectory_difference"] < PARITY["control_start_changed_by_1e-7"],     "the two samplers drift apart more than a start changed by 1e-7 does"
del teacher, whole, za, zb, zc
gc.collect()
torch.cuda.empty_cache()

if REPO:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")          # placed in this process's environment only
    # with the tiny engines' weights, so the upload of large files is checked too
    assert hub.ship(SMOKE, REPO, f"{PREFIX}/_smoke", "first capture: smoke run", ignore=NOT_SHIPPED), \
        "the smoke folder was not shipped: check the token and the repository before the long run"

# the license copy and a notice, shipped to the root of the results folder before any trained weights
LIC = hf_hub_download(repo_id="CompVis/stable-diffusion-license", repo_type="space", filename="license.txt", local_dir=f"{OUT}/license")
NOTICE = f"{OUT}/notice"
os.makedirs(NOTICE, exist_ok=True)
shutil.copyfile(LIC, f"{NOTICE}/LICENSE_STABLE_DIFFUSION_1_5.txt")       # byte for byte
lic_bytes = open(f"{NOTICE}/LICENSE_STABLE_DIFFUSION_1_5.txt", "rb").read()
print("license copy: sha256", hashlib.sha256(lic_bytes).hexdigest(), "|", len(lic_bytes), "bytes")
NOTICE_TEXT = (
    "# Notice: files made from Stable Diffusion 1.5\n\n"
    "The `engine.safetensors` and `engine_avg.safetensors` files in the folders beside this notice are band engines: reduced "
    "copies of the Stable Diffusion 1.5 U-Net (`stable-diffusion-v1-5/stable-diffusion-v1-5`, revision `451f4fe1`). Some start "
    "from that model's weights, and all were trained on its answers or in its latent space, so they are derivatives of that "
    "model.\n\n"
    "Stable Diffusion 1.5 is published under the CreativeML Open RAIL-M license. A copy of that license is in this folder "
    "(`LICENSE_STABLE_DIFFUSION_1_5.txt`); its use restrictions (its Attachment A) travel with these files as they do with the "
    "model.\n\n"
    "These are research artifacts of a measurement, not a finished model: each engine covers one stretch of the noise range and "
    "is read beside the original model.\n")
with open(f"{NOTICE}/NOTICE.md", "w", encoding="utf-8", newline="\n") as fh:
    fh.write(NOTICE_TEXT)
if REPO:
    assert hub.ship(NOTICE, REPO, PREFIX, "first capture: license copy and notice"), "the license copy and notice were not shipped"

# a timing on this card: 30 steps of each shape at batch 16, the trainer's per-step lines stamped (the card synchronized);
# the seconds a step = the median gap of steps 11 to 30
teacher = load_teacher(TEACHER, device=DEVICE, dtype=torch.float32)
source = LatentBatches(TRAIN, seed=0)
SEC = {}
for shape in ("tiny", "2222/nomid/3"):
    folder = f"{OUT}/timing/{shape.replace('/', '')}"
    shutil.rmtree(folder, ignore_errors=True)
    stamps = []
    def note(line):
        if line.startswith("step "):
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            stamps.append(time.time())
    train_band_engine(teacher, shape, "clean", source, steps=30, batch=16, seed=0, out=folder, text=False, autocast="bf16",
                      save_every=0, log_every=1, log=note)
    SEC[shape] = statistics.median([b - a for a, b in zip(stamps[9:-1], stamps[10:])])
    shutil.rmtree(folder, ignore_errors=True)
    print(f"{shape}: {SEC[shape]:.3f} s a training step at batch 16")
del teacher, source
gc.collect()
torch.cuda.empty_cache()

# each job's cost from those two numbers: one forward of 16 rows of the model ~ a quarter of a 2222/nomid/3 step (its
# step is the model's forward and the engine's forward and backward, about four forwards), of an engine a quarter of its
# own step; a read of one part on one band is 64 rows x 4 steps = 16 such forwards; a sampling of the 64 rows is 4 groups
# x 50 steps of 32 rows (guidance); every job also loads the model (about 40 s)
F = SEC["2222/nomid/3"] / 4
E = {s: SEC[s] / 4 for s in SEC}
def cost(args, kind):
    a = dict(x.split("=", 1) for x in args.split() if "=" in x)
    c = 40 + 3 * 16 * F                                            # the job's start and the model's answers on the three bands
    if kind == "born":
        c += 3 * 16 * (2 * F + (F if a.get("shape") == "full" else E["tiny"]))   # the teacher's copy, the full copy, the shape
    elif kind == "train":
        e = E[a["shape"]]
        c += int(a["steps"]) * SEC[a["shape"]] + 16 * e + (int(a["steps"]) // int(a["read_every"])) * 2 * 16 * e + 3 * 16 * e + 16 * F
    elif kind == "sample":
        c += 3 * 16 * F + 2 * 4 * 50 * 2 * F
    else:
        c += 3 * 16 * 4 * F + 2 * 4 * 50 * 2 * F
    return c

JOBS = job_list(RUNS)
COST = {t: cost(a, k) for t, a, _, k in JOBS}
assert len({t for t, *_ in JOBS}) == len(JOBS) and all(n in COST for _, _, ns, _ in JOBS for n in ns)
INSTRUMENTS = {"parity": PARITY, "seconds_a_step_batch16": SEC, "card": PARITY["card"], "training_rows": TRAIN_ROWS}
print(f"the job list: {len(JOBS)} jobs ({sum(k == 'train' for *_, k in JOBS)} trainings), about {sum(COST.values()) / 60:.0f} "
      f"projected minutes on this card with one runner")

One runner works the job list in order (the card is full with one training); a job waits for the jobs whose folders it reads. Each finished job folder is shipped at once to `REPO` with its engine files (only the resume checkpoint is left out; a training's folder also gets the model card of Stable Diffusion 1.5, which states its license), with the token from the notebook's secret `HF_TOKEN` (placed in this process's environment only; the jobs themselves run without it). At the end the summary, the roster, its log and the instrument checks are shipped to `_report`. The machine is given back (`RELEASE`) only when the report and every finished job were shipped; if an upload failed, the machine is kept and the cell names the jobs.

In [ ]:
# C3 - the job list, each finished job shipped with its engine files, a progress line a minute, the summary, the machine
from geolip.modular_diffusion import jobs
RELEASE = True                     # give the machine back at the end (only when every finished job and the report were shipped)
EXTRA = ""                         # more arguments for every job (empty in a real run)
PROGRESS_S = 60
kind = {t: k for t, _, _, k in JOBS}
needs = {t: n for t, _, n, _ in JOBS}
tags = [t for t, *_ in JOBS]
roster = f"{OUT}/roster.txt"
cap = lambda t: max(30, int(3 * COST[t] / 60))
with open(roster, "w", encoding="utf-8", newline="\n") as fh:
    fh.write("\n".join(f"{t} | {cap(t)} | geolip.modular_diffusion.run | capture {a} {COMMON}{EXTRA}" + (f" | needs={','.join(n)}" if n else "")
                       for t, a, n, _ in JOBS) + "\nEND\n")
mark = lambda t, m: os.path.exists(f"{RUNS}/{t}/{m}")
if REPO:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
ENGINE_FILES = ("engine.safetensors", "engine_avg.safetensors", "engine.json", "train.json")

def fetch_back():
    # on a new machine: finished jobs come back with their engine files, so nothing trains twice; a job comes back only
    # when what it reads came back too
    try:
        got = snapshot_download(repo_id=REPO, allow_patterns=[f"{PREFIX}/{t}/*" for t in tags], local_dir=f"{OUT}/hub_back")
    except Exception as e:
        print("nothing fetched back from the Hub:", repr(e)[:200])
        return
    for t in tags:
        src, dst = f"{got}/{PREFIX}/{t}", f"{RUNS}/{t}"
        if mark(t, "DONE") or not os.path.exists(f"{src}/result.json"):
            continue
        if kind[t] == "train" and not all(os.path.exists(f"{src}/{f}") for f in ENGINE_FILES):
            continue                                             # its engine files are not on the Hub: it is trained again
        if not all(mark(n, "DONE") for n in needs[t]):
            continue
        shutil.copytree(src, dst, dirs_exist_ok=True)
        open(f"{dst}/DONE", "w").write("fetched back from the Hub\n")
        open(f"{dst}/FROM_HUB", "w").write("fetched back from the Hub\n")

if REPO:
    fetch_back()
for t in tags:      # a claim left by an interrupted session would hold its job forever
    if mark(t, "CLAIM") and not (mark(t, "DONE") or mark(t, "FAILED")):
        os.remove(f"{RUNS}/{t}/CLAIM")
def after_job(tag, folder, ok):
    if ok and REPO:
        if hub.ship(folder, REPO, f"{PREFIX}/{tag}", f"first capture: {tag}", ignore=NOT_SHIPPED):
            open(f"{folder}/SHIPPED", "w").write("shipped to the Hub\n")      # kept across sessions on this machine
        else:
            print(f"{tag}: NOT SHIPPED (its folder stays on this machine: {folder})", flush=True)

shipped_or_fetched = lambda t: mark(t, "SHIPPED") or mark(t, "FROM_HUB")
for t in tags:      # a job that finished in an earlier session but whose upload failed: shipped again now
    if REPO and mark(t, "DONE") and not shipped_or_fetched(t):
        after_job(t, f"{RUNS}/{t}", True)

t0 = time.time()
left0 = sum(COST[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
print(f"start: {sum(mark(t, 'DONE') for t in tags)}/{len(tags)} jobs already done; about {left0 / 60:.0f} min to run; one runner")
runner = threading.Thread(target=jobs.run_roster, daemon=True,
                          kwargs=dict(roster=roster, out=RUNS, stop=f"{OUT}/STOP", log=f"{OUT}/roster.log", card_room_mib=None,
                                      ram_free_gb=4.0, max_hours=20.0, claim=True, after_job=after_job, wait_s=5.0,
                                      env={"OMP_NUM_THREADS": "4", "MKL_NUM_THREADS": "4"}))
runner.start()
while runner.is_alive():
    runner.join(PROGRESS_S)
    done, failed = sum(mark(t, "DONE") for t in tags), sum(mark(t, "FAILED") for t in tags)
    left = sum(COST[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
    spent = (time.time() - t0) / 60
    rate = (left0 - left) / 60 / spent if left0 > left and spent > 0 else 1.0     # projected minutes done a minute
    print(f"jobs done {done}/{len(tags)} (failed {failed}); {spent:.0f} min spent; about {left / 60 / rate:.0f} min left", flush=True)

summary = capture.summarize(RUNS)
print(summary)
REPORT = f"{OUT}/report"
os.makedirs(REPORT, exist_ok=True)
open(f"{REPORT}/summary.md", "w", encoding="utf-8").write(summary)
json.dump(INSTRUMENTS, open(f"{REPORT}/instruments.json", "w", encoding="utf-8"), indent=1)
for f in ("roster.txt", "roster.log", "runs/parity.json"):
    if os.path.exists(f"{OUT}/{f}"):
        shutil.copy(f"{OUT}/{f}", f"{REPORT}/{os.path.basename(f).replace('.log', '_log.txt')}")
for t in tags:      # the logs of the jobs that failed
    if mark(t, "FAILED"):
        for f in ("run.log", "log.txt"):
            if os.path.exists(f"{RUNS}/{t}/{f}"):
                shutil.copy(f"{RUNS}/{t}/{f}", f"{REPORT}/failed_{t}_{f.replace('.', '_')}.txt")
shipped = hub.ship(REPORT, REPO, f"{PREFIX}/_report", "first capture: summary, roster and checks") if REPO else None
unshipped = [t for t in tags if mark(t, "DONE") and not shipped_or_fetched(t)]
print("report shipped:", shipped, "| finished jobs not on the Hub:", ", ".join(unshipped) or "none")
if RELEASE and shipped and not unshipped:
    from google.colab import runtime
    runtime.unassign()
else:
    print("THE MACHINE IS KEPT:", "the report was not shipped" if not shipped else
          f"these finished jobs were not shipped: {', '.join(unshipped)} (run C3 again to retry)" if unshipped else "RELEASE is False")